In [1]:
import nltk
import numpy as np
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

nltk.download("punkt_tab")
nltk.download("wordnet")

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [2]:
# ===============================
# 1. Preprocessing
# ===============================
def preprocess(text):

    tokens = word_tokenize(text.lower())

    lemmatizer = WordNetLemmatizer()

    tokens = [
        lemmatizer.lemmatize(w)
        for w in tokens
        if w.isalpha()
    ]

    return tokens

In [3]:
# ===============================
# 2. Vocabulary
# ===============================
def build_vocab(tokens):

    vocab = list(set(tokens))

    word2idx = {w: i for i, w in enumerate(vocab)}
    idx2word = {i: w for i, w in enumerate(vocab)}

    return vocab, word2idx, idx2word


In [4]:
# ===============================
# 3. One Hot Encoding
# ===============================
def one_hot(word, word2idx, size):

    vec = np.zeros(size)

    if word in word2idx:
        vec[word2idx[word]] = 1

    return vec


In [5]:
# ===============================
# 4. Create Skip-Gram Data
# ===============================
def create_data(tokens, window=2):

    X, Y = [], []

    for i, target in enumerate(tokens):

        for j in range(max(0, i - window), min(len(tokens), i + window + 1)):

            if i != j:
                X.append(target)
                Y.append(tokens[j])

    return X, Y

In [6]:
# ===============================
# 5. Softmax
# ===============================
def softmax(x):

    e = np.exp(x - np.max(x))
    return e / np.sum(e)


In [7]:
# ===============================
# 6. Forward Propagation
# ===============================
def forward(x, W1, W2):

    h = np.dot(x, W1)        # Embedding
    y = np.dot(h, W2)

    return h, softmax(y)


In [8]:
# ===============================
# 7. Backward Propagation
# ===============================
def backward(x, h, y, t, W2):

    error = y - t

    dW2 = np.outer(h, error)
    dW1 = np.outer(x, np.dot(W2, error))

    return dW1, dW2

In [9]:
# ===============================
# 8. Train Word2Vec
# ===============================
def train(X, Y, word2idx, size, embed=10, lr=0.01, epochs=500):

    W1 = np.random.rand(size, embed)
    W2 = np.random.rand(embed, size)

    for e in range(epochs):

        loss = 0

        for target, context in zip(X, Y):

            x = one_hot(target, word2idx, size)
            t = one_hot(context, word2idx, size)

            h, y = forward(x, W1, W2)

            loss += -np.sum(t * np.log(y + 1e-9))

            dW1, dW2 = backward(x, h, y, t, W2)

            W1 -= lr * dW1
            W2 -= lr * dW2

        if e % 100 == 0:
            print("Epoch:", e, "Loss:", round(loss, 3))

    return W1, W2

In [10]:

# ===============================
# 9. Word Embeddings
# ===============================
def get_embeddings(W1, vocab):

    return {word: W1[i] for i, word in enumerate(vocab)}


# ===============================
# 10. Similar Words
# ===============================
def cosine(a, b):

    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


def similar_words(word, embeddings, top=3):

    scores = {}

    for w, vec in embeddings.items():
        if w != word:
            scores[w] = cosine(embeddings[word], vec)

    return sorted(scores, key=scores.get, reverse=True)[:top]


In [11]:
# ===============================
# 11. Main
# ===============================
def main():

    corpus = """
    Virat Kohli is a great cricket player.
    He scores many runs in every match.
    MS Dhoni is a famous captain.
    India won many cricket matches.
    Football players train hard every day.
    Messi scores beautiful goals.
    Ronaldo is a strong striker.
    """

    tokens = preprocess(corpus)
    print("Tokens:", tokens)

    vocab, word2idx, idx2word = build_vocab(tokens)
    size = len(vocab)

    X, Y = create_data(tokens)

    W1, W2 = train(X, Y, word2idx, size)

    embeddings = get_embeddings(W1, vocab)

    print("\nEmbedding Example:")
    print("cricket →", embeddings["cricket"])

    print("\nSimilar to 'cricket':")
    print(similar_words("cricket", embeddings))


# ===============================
# Run
# ===============================
if __name__ == "__main__":
    main()

Tokens: ['virat', 'kohli', 'is', 'a', 'great', 'cricket', 'player', 'he', 'score', 'many', 'run', 'in', 'every', 'match', 'm', 'dhoni', 'is', 'a', 'famous', 'captain', 'india', 'won', 'many', 'cricket', 'match', 'football', 'player', 'train', 'hard', 'every', 'day', 'messi', 'score', 'beautiful', 'goal', 'ronaldo', 'is', 'a', 'strong', 'striker']
Epoch: 0 Loss: 537.954
Epoch: 100 Loss: 322.151
Epoch: 200 Loss: 283.394
Epoch: 300 Loss: 278.162
Epoch: 400 Loss: 277.046

Embedding Example:
cricket → [ 0.92039482  1.02474648  0.43531451 -0.03845278  1.12796451 -0.40507256
 -0.59596974  0.15537205  0.78747932  2.71870573]

Similar to 'cricket':
['train', 'striker', 'great']
